# `DataCleaner` - Nettoyage des données agricoles

**Module :** `kadi.kidas.cleaner`  
**Classe :** `DataCleaner`

---

## Introduction

`DataCleaner` est la classe de nettoyage des données agricoles tabulaires. Elle offre :

- **Suppression des doublons** : exacts ou partiels (sous-ensembles de colonnes)
- **Imputation des NaN** : 4 stratégies — `mean`, `median`, `forward_fill`, `drop`
- **Détection des outliers** : 3 méthodes — IQR (Tukey), Z-score, MAD
- **Correction des dates** : normalisation des formats hétérogènes en `datetime64`
- **Standardisation du texte** : trim, suppression des accents, casse
- **Rapport détaillé** : journal de toutes les opérations effectuées

L'API est **chainable** : chaque méthode retourne le DataFrame modifié.

## 1. Création des données de démonstration

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import numpy as np
import pandas as pd

from kadi.kidas.cleaner import DataCleaner

# DataFrame avec tous les problèmes courants en AgriTech
df_brut = pd.DataFrame({
    'culture':         ['Maïs', 'Niébé', 'igname', 'MAÏS', None, 'sorgho', 'manioc', 'Maïs'],
    'commune':         ['Cotonou', 'Parakou', 'Bohicon', 'Cotonou', 'Kandi', 'Natitingou', 'Abomey', 'Cotonou'],
    'rendement_kg':    [1500, 800, 2000, 1500, np.nan, 1200, 3500, 1500],
    'prix_xof':        [350, 500, 250, 350, 300, 280, 180, 350],
    'outlier_col':     [1200, 1300, 1250, 1280, 15000, 1190, 1310, 1200],  # 15000 = outlier
    'date_recolte':    ['2024-01-15', '2024-02-20', '2024/03/10', '15-01-2024', None, '2024-05-01', '2024-06-15', '2024-01-15'],
    'notes':           ['  OK  ', 'Bon rendement!', 'C.A.S normal', 'Données OK', None, 'Bonne qualité', 'Stock épuisé', '  OK  '],
})

print(f"DataFrame brut : {len(df_brut)} lignes × {len(df_brut.columns)} colonnes")
df_brut

DataFrame brut : 8 lignes × 7 colonnes


,culture,commune,rendement_kg,prix_xof,outlier_col,date_recolte,notes
0,Maïs,Cotonou,1500.0,350,1200,2024-01-15,OK
1,Niébé,Parakou,800.0,500,1300,2024-02-20,Bon rendement!
2,igname,Bohicon,2000.0,250,1250,2024/03/10,C.A.S normal
3,MAÏS,Cotonou,1500.0,350,1280,15-01-2024,Données OK
4,NaN,Kandi,NaN,300,15000,NaN,NaN
5,sorgho,Natitingou,1200.0,280,1190,2024-05-01,Bonne qualité
6,manioc,Abomey,3500.0,180,1310,2024-06-15,Stock épuisé
7,Maïs,Cotonou,1500.0,350,1200,2024-01-15,OK


## 2. Initialisation de `DataCleaner`

In [2]:
# Le constructeur crée une copie interne du DataFrame
cleaner = DataCleaner(df_brut)
print(f"DataCleaner initialisé avec {cleaner._rapport['lignes_initiales']} lignes")

DataCleaner initialisé avec 8 lignes


## 3. `remove_duplicates()` - Suppression des doublons

In [3]:
cleaner = DataCleaner(df_brut)

# Suppression des doublons exacts (toutes colonnes)
df_sans_dup = cleaner.remove_duplicates()
print(f"Lignes avant : {len(df_brut)} → après : {len(df_sans_dup)}")
df_sans_dup

Lignes avant : 8 → après : 7


,culture,commune,rendement_kg,prix_xof,outlier_col,date_recolte,notes
0,Maïs,Cotonou,1500.0,350,1200,2024-01-15,OK
1,Niébé,Parakou,800.0,500,1300,2024-02-20,Bon rendement!
2,igname,Bohicon,2000.0,250,1250,2024/03/10,C.A.S normal
3,MAÏS,Cotonou,1500.0,350,1280,15-01-2024,Données OK
4,NaN,Kandi,NaN,300,15000,NaN,NaN
5,sorgho,Natitingou,1200.0,280,1190,2024-05-01,Bonne qualité
6,manioc,Abomey,3500.0,180,1310,2024-06-15,Stock épuisé


In [4]:
# Suppression basée sur un sous-ensemble de colonnes
cleaner2 = DataCleaner(df_brut)
df_dup_partiel = cleaner2.remove_duplicates(subset=['culture', 'commune'], keep='first')
print(f"Doublons (culture+commune) supprimés : {len(df_brut) - len(df_dup_partiel)}")

Doublons (culture+commune) supprimés : 1


## 4. `handle_missing_values()` - Traitement des valeurs manquantes

**Stratégies disponibles :** `'mean'`, `'median'`, `'forward_fill'`, `'drop'`

In [5]:
print(f"NaN dans 'rendement_kg' avant traitement : {df_brut['rendement_kg'].isna().sum()}")

# Stratégie 1 : imputation par la moyenne
cleaner = DataCleaner(df_brut)
df_mean = cleaner.handle_missing_values(strategy='mean', columns=['rendement_kg'])
print(f"Après 'mean'  : NaN restants = {df_mean['rendement_kg'].isna().sum()}")
print(f"Valeur imputée (moy) : {df_mean['rendement_kg'].iloc[4]:.1f} kg")

NaN dans 'rendement_kg' avant traitement : 1
Après 'mean'  : NaN restants = 0
Valeur imputée (moy) : 1714.3 kg


In [6]:
# Stratégie 2 : imputation par la médiane
cleaner = DataCleaner(df_brut)
df_median = cleaner.handle_missing_values(strategy='median', columns=['rendement_kg'])
print(f"Valeur imputée (médiane) : {df_median['rendement_kg'].iloc[4]:.1f} kg")

Valeur imputée (médiane) : 1500.0 kg


In [7]:
# Stratégie 3 : forward fill (propage la valeur précédente)
cleaner = DataCleaner(df_brut)
df_ffill = cleaner.handle_missing_values(strategy='forward_fill')
print(f"NaN restants (ffill) : {df_ffill.isna().sum().sum()}")

NaN restants (ffill) : 0


In [8]:
# Stratégie 4 : suppression des lignes avec NaN
cleaner = DataCleaner(df_brut)
df_drop = cleaner.handle_missing_values(strategy='drop')
print(f"Lignes supprimées : {len(df_brut) - len(df_drop)}")

Lignes supprimées : 1


## 5. `remove_outliers()` - Détection et suppression des outliers

**Méthodes disponibles :** `'iqr'` (Tukey), `'zscore'`, `'mad'` (robuste)

In [9]:
# Méthode IQR (règle de Tukey : threshold=1.5 standard)
cleaner = DataCleaner(df_brut)
df_propre, df_outliers = cleaner.remove_outliers(method='iqr', threshold=1.5, columns=['outlier_col'])

print(f"Outliers détectés (IQR) : {len(df_outliers)}")
print(f"Valeur identifiée comme outlier : {df_outliers['outlier_col'].values}")

Outliers détectés (IQR) : 1
Valeur identifiée comme outlier : [15000]


In [10]:
# Méthode Z-score (seuil = 3 déviations standard)
cleaner = DataCleaner(df_brut)
df_propre_z, df_out_z = cleaner.remove_outliers(method='zscore', threshold=2.5, columns=['outlier_col'])
print(f"Outliers (Z-score, threshold=2.5) : {len(df_out_z)}")

Outliers (Z-score, threshold=2.5) : 1


In [11]:
# Méthode MAD (Median Absolute Deviation, robuste aux extrêmes)
cleaner = DataCleaner(df_brut)
df_propre_mad, df_out_mad = cleaner.remove_outliers(method='mad', threshold=3.0, columns=['outlier_col'])
print(f"Outliers (MAD, threshold=3.0) : {len(df_out_mad)}")

Outliers (MAD, threshold=3.0) : 1


## 6. `fix_dates()` - Normalisation des formats de dates

In [12]:
print("Formats de dates brutes :")
print(df_brut['date_recolte'].dropna().tolist())

cleaner = DataCleaner(df_brut)
df_dates = cleaner.fix_dates(columns=['date_recolte'])

print(f"\nType après normalisation : {df_dates['date_recolte'].dtype}")
print("Valeurs converties :")
df_dates[['date_recolte', 'culture']].dropna()

Formats de dates brutes :
['2024-01-15', '2024-02-20', '2024/03/10', '15-01-2024', '2024-05-01', '2024-06-15', '2024-01-15']

Type après normalisation : datetime64[us]
Valeurs converties :


,date_recolte,culture
0,2024-01-15,Maïs
1,2024-02-20,Niébé
2,2024-03-10,igname
3,2024-01-15,MAÏS
5,2024-05-01,sorgho
6,2024-06-15,manioc
7,2024-01-15,Maïs


## 7. `standardize_text()` - Standardisation du texte

In [13]:
print("Textes bruts dans 'culture' :")
print(df_brut['culture'].tolist())

# Conversion en minuscules + suppression des espaces et accents
cleaner = DataCleaner(df_brut)
df_text = cleaner.standardize_text(columns=['culture', 'commune'], case='lower')

print("\nAprès standardisation (lower) :")
print(df_text['culture'].tolist())

Textes bruts dans 'culture' :
['Maïs', 'Niébé', 'igname', 'MAÏS', nan, 'sorgho', 'manioc', 'Maïs']

Après standardisation (lower) :
['mais', 'niebe', 'igname', 'mais', nan, 'sorgho', 'manioc', 'mais']


In [14]:
# Titre (chaque mot commence par une majuscule)
cleaner = DataCleaner(df_brut)
df_title = cleaner.standardize_text(columns=['commune'], case='title')
print("Communes en titre :")
print(df_title['commune'].tolist())

Communes en titre :
['Cotonou', 'Parakou', 'Bohicon', 'Cotonou', 'Kandi', 'Natitingou', 'Abomey', 'Cotonou']


## 8. `remove_special_chars()` - Suppression des caractères spéciaux

In [15]:
print("Notes brutes :")
print(df_brut['notes'].tolist())

cleaner = DataCleaner(df_brut)
df_notes = cleaner.remove_special_chars(columns=['notes'], keep_chars='-')

print("\nNotes nettoyées :")
print(df_notes['notes'].tolist())

Notes brutes :
['  OK  ', 'Bon rendement!', 'C.A.S normal', 'Données OK', nan, 'Bonne qualité', 'Stock épuisé', '  OK  ']

Notes nettoyées :
['OK', 'Bon rendement', 'CAS normal', 'Donnes OK', nan, 'Bonne qualit', 'Stock puis', 'OK']


## 9. `detect_inconsistent_decimals()` - Détection des séparateurs décimaux mixtes

In [16]:
# Données avec mélange de '.' et ',' comme séparateur décimal
df_decimales = pd.DataFrame({
    'prix_str': ['1.500', '2,300', '1.800', '3,150', '2.000'],
    'poids_str': ['50.5', '48.0', '51,3', '49.8', '50.1'],
})

cleaner = DataCleaner(df_decimales)
rapport_dec = cleaner.detect_inconsistent_decimals(columns=['prix_str', 'poids_str'])

print("Rapport des séparateurs décimaux :")
for col, info in rapport_dec.items():
    print(f"\n  Colonne '{col}' :")
    for cle, val in info.items():
        print(f"    {cle:15s} : {val}")

Mélange de séparateurs décimaux détecté dans 'prix_str' (3 points, 2 virgules).
Mélange de séparateurs décimaux détecté dans 'poids_str' (4 points, 1 virgules).


Rapport des séparateurs décimaux :

  Colonne 'prix_str' :
    has_dot         : True
    has_comma       : True
    mixed           : True
    count_dot       : 3
    count_comma     : 2

  Colonne 'poids_str' :
    has_dot         : True
    has_comma       : True
    mixed           : True
    count_dot       : 4
    count_comma     : 1


## 10. Pipeline chainé - Exemple complet AgriTech

In [17]:
# Pipeline de nettoyage complet enchaîné
cleaner = DataCleaner(df_brut)

df_propre = cleaner.remove_duplicates()
df_propre = cleaner.handle_missing_values(strategy='mean', columns=['rendement_kg', 'prix_xof'])
df_propre = cleaner.standardize_text(columns=['culture', 'commune'], case='lower')
df_propre = cleaner.fix_dates(columns=['date_recolte'])
df_propre = cleaner.remove_special_chars(columns=['notes'])
df_propre, _ = cleaner.remove_outliers(method='iqr', columns=['outlier_col'])

print(f"DataFrame nettoyé : {len(df_propre)} lignes × {len(df_propre.columns)} colonnes")
df_propre

DataFrame nettoyé : 6 lignes × 7 colonnes


,culture,commune,rendement_kg,prix_xof,outlier_col,date_recolte,notes
0,mais,cotonou,1500.0,350,1200,2024-01-15,OK
1,niebe,parakou,800.0,500,1300,2024-02-20,Bon rendement
2,igname,bohicon,2000.0,250,1250,2024-03-10,CAS normal
3,mais,cotonou,1500.0,350,1280,2024-01-15,Donnes OK
5,sorgho,natitingou,1200.0,280,1190,2024-05-01,Bonne qualit
6,manioc,abomey,3500.0,180,1310,2024-06-15,Stock puis


## 11. `get_cleaning_report()` - Rapport des opérations effectuées

In [18]:
rapport = cleaner.get_cleaning_report()

print("=== Rapport de Nettoyage ===")
print(f"Lignes initiales       : {rapport['lignes_initiales']}")
print(f"Lignes finales         : {rapport['lignes_finales']}")
print(f"Doublons supprimés     : {rapport['doublons_supprimes']}")
print(f"NaN traités            : {rapport['nan_traites']}")
print(f"Outliers supprimés     : {rapport['outliers_detectes']}")
print(f"Dates corrigées        : {rapport['dates_corrigees']}")
print(f"\nOpérations effectuées ({len(rapport['operations'])}) :")
for op in rapport['operations']:
    print(f"  - {op['operation']}")

=== Rapport de Nettoyage ===
Lignes initiales       : 8
Lignes finales         : 6
Doublons supprimés     : 1
NaN traités            : 1
Outliers supprimés     : 1
Dates corrigées        : 6

Opérations effectuées (6) :
  - remove_duplicates
  - handle_missing_values
  - standardize_text
  - fix_dates
  - remove_special_chars
  - remove_outliers


## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `remove_duplicates(subset, keep)` | Supprime les lignes dupliquées |
| `handle_missing_values(strategy, columns)` | Impute les NaN (mean/median/ffill/drop) |
| `remove_outliers(method, threshold, columns)` | Détecte les outliers (IQR/Z-score/MAD) |
| `fix_dates(columns)` | Normalise les formats de dates en datetime64 |
| `standardize_text(columns, case)` | Trim, accents, casse |
| `remove_special_chars(columns, keep_chars)` | Supprime les caractères spéciaux |
| `detect_inconsistent_decimals(columns)` | Détecte les mélanges `.`/`,` |
| `get_cleaning_report()` | Journal complet des opérations |
